# 1.5 Weather Cleanup

In this notebook we clean the weather data from https://open-meteo.com/en/docs (downloaded on 28.05.2026).

### Data Requirements:

This notebook requires the following file: 

`01_02_weather_hourly.parquet` — local path: `data/weather/01_02_weather_hourly.parquet` — Sciebo path: `data_parquet/weather/01_02_weather_hourly.parquet`

Sample data is provided in the appropriate locations for testing purposes only. It will not produce meaningful results.

In [1]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Import packages                         #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import pandas as pd
import matplotlib.pyplot as plt
import math
import numpy as np
from scipy.stats import zscore
from shapely.geometry import Polygon, Point

import geopandas as gpd
import h3

# reset working dir
import os
from pathlib import Path

In [2]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Reset working directory                 #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

import __main__
_nb = getattr(__main__, "__vsc_ipynb_file__", None) or os.environ.get("JPY_SESSION_NAME")
_start = Path(_nb).resolve().parent if _nb else Path.cwd()
os.chdir(next(p for p in [_start, *_start.parents] if (p / "pyproject.toml").exists()))
print(f"Working directory: {os.getcwd()}")

Working directory: /Users/hendrik/Coding/Master/AAA


In [3]:
# # # # # # # # # # # # # # # # # # # # # #
#                                         #
# Load weather data                       #
#                                         #
# # # # # # # # # # # # # # # # # # # # # #

weather = pd.read_parquet("data/weather/01_02_weather_hourly.parquet")
print(weather.shape)
weather.head()

(17568, 15)


,date,temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,rain,snowfall,snow_depth,surface_pressure,cloud_cover,wind_speed_10m,wind_speed_100m,is_day,sunshine_duration,direct_radiation
0,2024-01-01 00:00:00+00:00,0.80,80.643494,-5.467376,0.0,0.0,0.0,0.01,988.042114,100.0,26.282465,37.142860,0.0,0.0,0.0
1,2024-01-01 01:00:00+00:00,0.45,81.492714,-5.740202,0.0,0.0,0.0,0.01,988.489746,100.0,25.570139,36.557289,0.0,0.0,0.0
2,2024-01-01 02:00:00+00:00,0.20,81.760498,-5.956933,0.0,0.0,0.0,0.01,989.044922,100.0,25.161400,35.685421,0.0,0.0,0.0
3,2024-01-01 03:00:00+00:00,0.20,81.760498,-6.078120,0.0,0.0,0.0,0.01,989.530151,100.0,25.992401,37.854813,0.0,0.0,0.0
4,2024-01-01 04:00:00+00:00,0.25,81.767311,-6.382498,0.0,0.0,0.0,0.01,990.020630,100.0,28.467327,41.230610,0.0,0.0,0.0


In [4]:
weather.info()

<class 'pandas.DataFrame'>
RangeIndex: 17568 entries, 0 to 17567
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype              
---  ------                --------------  -----              
 0   date                  17568 non-null  datetime64[ms, UTC]
 1   temperature_2m        17568 non-null  float32            
 2   relative_humidity_2m  17568 non-null  float32            
 3   apparent_temperature  17568 non-null  float32            
 4   precipitation         17568 non-null  float32            
 5   rain                  17568 non-null  float32            
 6   snowfall              17568 non-null  float32            
 7   snow_depth            17568 non-null  float32            
 8   surface_pressure      17568 non-null  float32            
 9   cloud_cover           17568 non-null  float32            
 10  wind_speed_10m        17568 non-null  float32            
 11  wind_speed_100m       17568 non-null  float32            
 12  is_day         

In [5]:
display(weather)

,date,temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,rain,snowfall,snow_depth,surface_pressure,cloud_cover,wind_speed_10m,wind_speed_100m,is_day,sunshine_duration,direct_radiation
0,2024-01-01 00:00:00+00:00,0.80,80.643494,-5.467376,0.0,0.0,0.00,0.01,988.042114,100.0,26.282465,37.142860,0.0,0.0,0.0
1,2024-01-01 01:00:00+00:00,0.45,81.492714,-5.740202,0.0,0.0,0.00,0.01,988.489746,100.0,25.570139,36.557289,0.0,0.0,0.0
2,2024-01-01 02:00:00+00:00,0.20,81.760498,-5.956933,0.0,0.0,0.00,0.01,989.044922,100.0,25.161400,35.685421,0.0,0.0,0.0
3,2024-01-01 03:00:00+00:00,0.20,81.760498,-6.078120,0.0,0.0,0.00,0.01,989.530151,100.0,25.992401,37.854813,0.0,0.0,0.0
4,2024-01-01 04:00:00+00:00,0.25,81.767311,-6.382498,0.0,0.0,0.00,0.01,990.020630,100.0,28.467327,41.230610,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17563,2026-01-01 19:00:00+00:00,-8.60,64.735161,-13.260756,0.2,0.0,0.14,0.03,986.800049,100.0,7.748368,11.183201,1.0,0.0,7.0
17564,2026-01-01 20:00:00+00:00,-8.45,65.036278,-12.784360,0.1,0.0,0.07,0.03,985.847717,100.0,5.591600,7.952660,1.0,0.0,1.0
17565,2026-01-01 21:00:00+00:00,-8.15,63.796635,-12.443245,0.0,0.0,0.00,0.03,985.882141,100.0,5.330554,7.421590,1.0,0.0,5.0
17566,2026-01-01 22:00:00+00:00,-8.10,64.330788,-12.297367,0.0,0.0,0.00,0.03,985.112427,98.0,4.735060,7.102591,1.0,0.0,2.0


weather timestamps are UTC-aware; convert to Chicago local time and strip tz:

In [6]:
weather["date"] = (
    pd.to_datetime(weather["date"], utc=True)
    .dt.tz_convert("America/Chicago")
    .dt.tz_localize(None)
)
weather = weather.sort_values("date")

Check for NaN values in weather:

In [7]:
weather.isna().sum().to_frame(name="Null Count").assign(
    Null_Percent=lambda x: (x["Null Count"] / len(weather) * 100).round(2)
)

,Null Count,Null_Percent
date,0,0.0
temperature_2m,0,0.0
relative_humidity_2m,0,0.0
apparent_temperature,0,0.0
precipitation,0,0.0
rain,0,0.0
snowfall,0,0.0
snow_depth,0,0.0
surface_pressure,0,0.0
cloud_cover,0,0.0


check the time range:

In [8]:
print("Start Timestamp range:", weather["date"].min(), "to", weather["date"].max())

Start Timestamp range: 2023-12-31 18:00:00 to 2026-01-01 17:00:00


In [9]:
#limit the data to 2025 only

weather = weather[ (weather["date"]>= "2025-01-01") & (weather["date"] < "2026-01-01") ].reset_index()

print("Start Timestamp range:", weather["date"].min(), "to", weather["date"].max())

Start Timestamp range: 2025-01-01 00:00:00 to 2025-12-31 23:00:00


In [10]:
weather.info()

<class 'pandas.DataFrame'>
RangeIndex: 8760 entries, 0 to 8759
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   index                 8760 non-null   int64         
 1   date                  8760 non-null   datetime64[ms]
 2   temperature_2m        8760 non-null   float32       
 3   relative_humidity_2m  8760 non-null   float32       
 4   apparent_temperature  8760 non-null   float32       
 5   precipitation         8760 non-null   float32       
 6   rain                  8760 non-null   float32       
 7   snowfall              8760 non-null   float32       
 8   snow_depth            8760 non-null   float32       
 9   surface_pressure      8760 non-null   float32       
 10  cloud_cover           8760 non-null   float32       
 11  wind_speed_10m        8760 non-null   float32       
 12  wind_speed_100m       8760 non-null   float32       
 13  is_day                8760 no

check for duplicates

In [11]:
print("Amount of duplicated rows: " + str(weather.duplicated().any().sum()))
print("Amount of duplicated dates: " + str(weather.date.duplicated().any().sum()))

Amount of duplicated rows: 0
Amount of duplicated dates: 1


In [12]:
weather[weather.date.duplicated()]

,index,date,temperature_2m,relative_humidity_2m,apparent_temperature,precipitation,rain,snowfall,snow_depth,surface_pressure,cloud_cover,wind_speed_10m,wind_speed_100m,is_day,sunshine_duration,direct_radiation
7321,16111,2025-11-02 01:00:00,6.05,93.95443,4.463691,0.0,0.0,0.0,0.0,992.093323,53.0,2.747581,6.948093,0.0,0.0,0.0


The duplicated date comes from the transition from daylight saving time, so its not erroneous

Check for gaps in the hourly data:

In [13]:
# create an hourly timerange as an attribute to compare with date
full_range = pd.date_range(
    start=weather.date.min(),
    end=weather.date.max(),
    freq="h"
)

In [14]:
missing_timestamps = full_range.difference(weather.date)
print("amount of gaps in 2025: " + str(len(missing_timestamps)))
missing_timestamps

amount of gaps in 2025: 1


DatetimeIndex(['2025-03-09 02:00:00'], dtype='datetime64[ms]', freq='h')

In [15]:
weather.to_parquet("data/weather/01_05_weather_clean.parquet", index=False)

Again, daylight saving time is the reason for the missing date.